# Research foundations: saved inputs to a research dataset

For researchers with saved OKX historical L2 and Bitfinex R0 recordings, this
walkthrough answers five questions:

1. What visible liquidity can I reconstruct, and which clocks are available?
2. How do separate saved inputs flow through admission into one reproducible dataset?
3. Which revisions and membership assertions were known at a decision cutoff?
4. Can I consume bounded price batches without losing controls or replay history?
5. Does the reference reader meet the measured one-hour trade-inspection workload?

The book exploration uses real saved inputs. The dataset walkthrough uses **small
synthetic trades**, because published LOB adapters are planned for MR 25. Everything
here runs offline, without credentials. If inputs are missing, first use
[market-data-capture.ipynb](market-data-capture.ipynb); see the
[operations guide](notebook-operations.md) for installation and pinned input paths.

Restart the kernel and **Run All in order**. Expand the collapsed setup cells only
when needed. These observations do not replace the [frozen P1 study](../docs/research/p1-baseline.md)
or establish coverage, fills or a strategy result.

## 1. Identify this execution

The installed package must match the checkout and pinned interpreter. The table
records code, configuration and notebook fingerprints for comparing later runs.

In [ ]:
# standard library imports
import hashlib
import json
import os
import platform
import subprocess
import sys
from datetime import UTC, datetime, timedelta
from decimal import Decimal
from html import escape
from importlib.metadata import distributions, version
from pathlib import Path

# third party library imports
from IPython.display import HTML, display

# project library imports
import alpha_research
from alpha_research.data.live.config import CaptureConfig, CaptureRequest
from alpha_research.data.live.journal import CaptureJournal, CaptureRecord
from alpha_research.data.live.replay import CaptureReplay
from alpha_research.data.raw_objects import RawObject
from alpha_research.data.venues.bitfinex.book import BitfinexBook
from alpha_research.data.venues.okx.history_config import OKXArchiveRequest, OKXHistoryConfig
from alpha_research.data.venues.okx.history_store import OKXArchiveStore
from alpha_research.serialization import DomainCodec

root = Path.cwd().resolve()
if root.name == "examples":
    root = root.parent
assert (root / "configs/history/okx-btc-usdt.json").is_file(), (
    "Run from the repository root or examples/."
)
notebook_path = Path(
    os.environ.get("ALPHA_RESEARCH_NOTEBOOK_PATH", root / "examples/research-foundations.ipynb")
).resolve()


class NotebookDisplay:
    """Render small tables for the exploratory notebook scenarios."""

    @staticmethod
    def show_table(headers: tuple[str, ...], rows: list[tuple[object, ...]]) -> None:
        """Display a small escaped table without adding a dataframe dependency.

        Args:
            headers: Column labels.
            rows: Exact or presentation values to render as text.
        """
        heading = "".join(
            f'<th style="text-align:left;padding:6px">{escape(label)}</th>' for label in headers
        )
        body = "".join(
            "<tr>"
            + "".join(
                f'<td style="padding:6px;vertical-align:top">{escape(str(value))}</td>'
                for value in row
            )
            + "</tr>"
            for row in rows
        )
        display(HTML(f"<table><thead><tr>{heading}</tr></thead><tbody>{body}</tbody></table>"))

In [ ]:
assert platform.python_version() == (root / ".python-version").read_text().strip() == "3.14.7"
installed = Path(alpha_research.__file__).resolve().parent
assert "site-packages" in installed.parts and installed != root / "src/alpha_research"
source_root = root / "src/alpha_research"
runtime_files = sorted(
    path
    for path in source_root.rglob("*")
    if path.is_file() and path.suffix in {".py", ".json", ".typed"}
)
assert runtime_files
runtime_digest = hashlib.sha256()
for source_file in runtime_files:
    relative = source_file.relative_to(source_root)
    source_bytes = source_file.read_bytes()
    assert source_bytes == (installed / relative).read_bytes(), f"Stale installed file: {relative}"
    runtime_digest.update(str(relative).encode() + b"\0" + source_bytes + b"\0")

commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=root, text=True).strip()
branch = subprocess.check_output(["git", "branch", "--show-current"], cwd=root, text=True).strip()
working_status = subprocess.check_output(["git", "status", "--short"], cwd=root, text=True).strip()
candidate_diff = subprocess.check_output(
    [
        "git",
        "diff",
        "--binary",
        "HEAD",
        "--",
        "src",
        "configs",
        "pyproject.toml",
        "uv.lock",
        ".python-version",
    ],
    cwd=root,
)
notebook_document = json.loads(notebook_path.read_text())
notebook_source = [
    (cell["cell_type"], "".join(cell["source"])) for cell in notebook_document["cells"]
]
notebook_digest = hashlib.sha256(
    json.dumps(notebook_source, ensure_ascii=False, separators=(",", ":")).encode()
).hexdigest()
identity = {
    "notebook_path": str(notebook_path),
    "attempt_id": os.environ.get("ALPHA_RESEARCH_ATTEMPT_ID", "interactive: unsnapshotted"),
    "commit": commit,
    "branch": branch,
    "working_tree": "changed" if working_status else "clean",
    "python": platform.python_version(),
    "platform": platform.platform(),
    "kernel": sys.executable,
    "package": str(installed),
    "verified_runtime_files": len(runtime_files),
    "runtime_sha256": runtime_digest.hexdigest(),
    "runtime_input_diff_sha256": hashlib.sha256(candidate_diff).hexdigest(),
    "uv_lock_sha256": hashlib.sha256((root / "uv.lock").read_bytes()).hexdigest(),
    "okx_config_sha256": hashlib.sha256(
        (root / "configs/history/okx-btc-usdt.json").read_bytes()
    ).hexdigest(),
    "notebook_source_sha256": notebook_digest,
    "notebook_tooling": {
        item.metadata["Name"]: version(item.metadata["Name"])
        for item in distributions()
        if item.metadata["Name"] in {"nbclient", "nbformat", "ipykernel"}
    },
}
NotebookDisplay.show_table(("Identity", "Observed value"), list(identity.items()))

## 2. Select the saved books

Pin `ALPHA_OKX_ARCHIVE_DIRECTORY` and `ALPHA_BITFINEX_CAPTURE_DIRECTORY` to repeat
an observation. Otherwise, this selects the latest completed input matching each
maintained configuration. The printed paths identify this convenience selection;
it is not yet a published research dataset. Invalid or absent inputs fail visibly.

In [ ]:
codec = DomainCodec()
okx_config = OKXHistoryConfig.read(root / "configs/history/okx-btc-usdt.json")
bitfinex_config = CaptureConfig.read(root / "configs/live/bitfinex-btc-usd.json")
selected = {}
selection_rows = []
for venue, folder, variable, request_type, config in (
    ("okx", "data/okx/history", "ALPHA_OKX_ARCHIVE_DIRECTORY", OKXArchiveRequest, okx_config),
    (
        "bitfinex",
        "data/bitfinex",
        "ALPHA_BITFINEX_CAPTURE_DIRECTORY",
        CaptureRequest,
        bitfinex_config,
    ),
):
    override = os.environ.get(variable)
    candidates = []
    paths = (
        [Path(override).expanduser()]
        if override
        else [p.parent for p in (root / folder).glob("*/manifest.json")]
    )
    for path in paths:
        request = codec.decode_json((path / "request.json").read_bytes(), request_type)
        if request.config != config:
            if override:
                raise ValueError(f"{variable} differs from the maintained selection")
            continue
        descriptor = codec.decode_json((path / "manifest.json").read_bytes(), RawObject)
        candidates.append((descriptor.ts_ingested, str(path.resolve()), request, descriptor))
    if not candidates:
        raise FileNotFoundError(f"No matching {venue} input; run market-data-capture.ipynb first")
    _, location, request, descriptor = max(candidates, key=lambda item: (item[0], item[1]))
    selected[venue] = (Path(location), request, descriptor)
    selection_rows.extend(
        [
            (
                venue,
                "Selection",
                "explicit directory" if override else "latest matching saved input",
            ),
            (venue, "Directory", location),
            (venue, "Raw SHA-256", descriptor.sha256),
            (venue, "Request identity", request.identity),
            (venue, "Original implementation SHA-256", request.implementation_identity),
        ]
    )
NotebookDisplay.show_table(("Source", "Evidence", "Value"), selection_rows)

## 3. Reconstruct and inspect visible liquidity

**OKX BTC-USDT:** verify the historical archive for 8 September 2026 and reconstruct
its first **10,000 records**. L2 aggregates size at each price; it supplies no
individual order identity. This reader returns a summary, not a historical price table.

In [ ]:
okx_directory, okx_request, okx_descriptor = selected["okx"]
history = OKXArchiveStore(okx_directory).replay()
assert history["source_sequence"] is None and history["gaps"] is None
epoch = datetime(1970, 1, 1, tzinfo=UTC)
history_rows = [
    (key, history[key])
    for key in (
        "symbol",
        "representation",
        "records",
        "snapshots",
        "updates",
        "levels",
        "depth",
        "state_available",
        "continuity",
        "stop_reason",
        "entire_archive_replayed",
    )
]
for key in ("first_source_ms", "last_source_ms"):
    value = history[key]
    history_rows.append(
        (
            key.replace("_ms", "_utc"),
            None if value is None else epoch + timedelta(milliseconds=int(value)),
        )
    )
NotebookDisplay.show_table(("Historical observation", "Result"), history_rows)

`gaps=None` means continuity is unknown because the archive has no source sequence
numbers. An available reconstructed state does not prove uninterrupted coverage.

**Bitfinex BTC/USD:** replay the saved bounded R0 recording, then inspect the nearest
three individual orders on each side. Prices and spreads are USD per BTC; sizes
are BTC. Configured depth 25 and the number of retained orders are different measures.
The spread describes this recording and does not account for execution costs.

In [ ]:
bitfinex_directory, bitfinex_request, bitfinex_descriptor = selected["bitfinex"]
replay = CaptureReplay.read(bitfinex_directory)
summary = replay.summary()
assert isinstance(replay.book, BitfinexBook)
live_rows = [
    (key, summary[key])
    for key in (
        "symbol",
        "messages",
        "snapshots",
        "updates",
        "orders",
        "depth",
        "gaps",
        "synchronized",
        "source_sequence",
        "last_source_time",
    )
]
for key in ("started_ns", "finished_ns"):
    live_rows.append(
        (key.replace("_ns", "_utc"), epoch + timedelta(microseconds=summary[key] // 1000))
    )
NotebookDisplay.show_table(("Recorded observation", "Result"), live_rows)
assert summary["synchronized"] and not summary["gaps"], (
    "This sample is unsuitable for the uninterrupted-book preview; inspect its gaps"
)

preview_rows = []
best = {}
for side in ("buy", "sell"):
    orders = sorted(
        (order for order in replay.book.orders.values() if order.side == side),
        key=lambda order: order.price,
        reverse=side == "buy",
    )
    assert orders, f"No observed {side} side"
    best[side] = orders[0].price
    for order in orders[:3]:
        assert isinstance(order.price, Decimal) and isinstance(order.size, Decimal)
        preview_rows.append((side, order.order_id, str(order.price), str(order.size)))
NotebookDisplay.show_table(("Side", "Visible order ID", "USD / BTC", "BTC"), preview_rows)
NotebookDisplay.show_table(
    ("Measure", "USD / BTC"),
    [
        ("Best bid", best["buy"]),
        ("Best ask", best["sell"]),
        ("Observed spread", best["sell"] - best["buy"]),
    ],
)

## 4. Check clocks and repeatability

OKX supplies historical event timestamps but no historical receipt clock. Bitfinex
R0 supplies local receipt order but no exchange timestamp in these messages. Their
quote currencies, times and depths differ; **USD is not USDT** and these are not
simultaneous price observations. UTC display truncates journal nanoseconds to microseconds.

Replaying the same bytes must reproduce the same state trace. The final control
injects a disconnect into a separate replay: the book becomes unavailable until
resynchronization, instead of carrying stale quotes across the gap.

In [ ]:
NotebookDisplay.show_table(
    ("Sample", "Source clock", "Local receipt evidence", "Saved bytes"),
    [
        (
            "OKX historical BTC-USDT",
            "Unix milliseconds in records; continuity unknown",
            f"Archive received {history['download_received']}; per-record receipt absent",
            history["saved_bytes"],
        ),
        (
            "Bitfinex recorded BTC/USD",
            "Absent in R0 messages",
            f"Journal nanoseconds {summary['started_ns']} to {summary['finished_ns']}",
            bitfinex_descriptor.size_bytes,
        ),
    ],
)

again = CaptureReplay.read(bitfinex_directory)
assert again.summary()["trace_identity"] == summary["trace_identity"]
exercise = CaptureReplay(bitfinex_config)
for record in CaptureJournal(bitfinex_directory, bitfinex_request).records():
    if record.kind == "end":
        break
    exercise.apply(record)
exercise.apply(
    CaptureRecord(
        kind="gap",
        epoch=exercise.epoch,
        received_ns=exercise.finished_ns,
        reason="synthetic_disconnect",
    )
)
assert not exercise.book.ready
NotebookDisplay.show_table(
    ("Check", "Observed result"),
    [
        ("Repeated state trace", summary["trace_identity"]),
        ("Injected disconnect", f"Book usable: {exercise.book.ready}"),
    ],
)

## 5. Prepare separate inputs for one dataset

The archive workflow is:

**saved acquisitions + per-input evidence → admission and assembly → atomic publication → verified reading**

`DatasetInput` carries each acquisition's quality policy, eligibility history and
correction evidence. Assembly retains the source, stream and instrument of every
partition. It is the composition boundary for future combinations of LOB sources;
those streams will remain separate within the dataset.

Today the installed saved-input adapter supports trade archives. This offline
example creates two **synthetic** instrument inputs using that adapter, with an
intentional duplicate in the first input. The CSV construction below is fixture
setup; the publication calls use the generic API. Actual LOB publication waits for
MR 25. The real OKX and Bitfinex recordings above are not published here.

In [ ]:
# standard library imports
from tempfile import TemporaryDirectory
from zipfile import ZIP_DEFLATED, ZipFile, ZipInfo

# third party library imports
from msgspec.structs import replace

# project library imports
from alpha_research.configuration import Configuration
from alpha_research.data.asof import AsOfSelector
from alpha_research.data.datasets import DatasetInput
from alpha_research.data.eligibility import EligibilityAssertions
from alpha_research.data.manifests import AcquisitionManifest
from alpha_research.data.quality import QualityPolicy
from alpha_research.data.raw_store import RawStore
from alpha_research.data.requests import SourceRequest
from alpha_research.data.snapshots import SnapshotStore
from alpha_research.pipelines.dataset import publish_dataset
from alpha_research.reference_data import CatalogSnapshot

archive_catalog = codec.decode_json(
    (root / "configs/reference/binance-archive-spot.json").read_bytes(), CatalogSnapshot
)
archive_run = Configuration.read(root / "configs/data-jobs/binance-archive.json").resolve(
    "btc-archive", archive_catalog
)
eligibility = EligibilityAssertions.read(
    root / "tests/fixtures/mr12/eligibility.json", catalog=archive_catalog
)
start = archive_run.job.start
policy = QualityPolicy(use="exploratory", known_at=start, max_gap=timedelta(hours=1))
max_bytes = 262_144
acquired_at = datetime(2026, 9, 17, tzinfo=UTC)
workspace = TemporaryDirectory(prefix="research-dataset-")
local = Path(workspace.name)
raw_directory, destination = local / "raw", local / "snapshots"
raw_directory.mkdir()
destination.mkdir()
store = RawStore(raw_directory)
snapshots = SnapshotStore(destination, store)
selections = []
for index, instrument in enumerate(archive_catalog.instruments):
    run = replace(archive_run, instruments=(instrument,))
    request = SourceRequest.partition(run)[0]
    source = local / "source.zip"
    member = ZipInfo(f"{instrument.symbol}-trades-{request.start.date()}.csv")
    member.compress_type = ZIP_DEFLATED
    with ZipFile(source, "w") as archive:
        archive.writestr(member, b"12,100.00,2.0,200,1717200000001,true,true\n" * (2 - index))
    origin = RawObject.from_file(
        source,
        request=request,
        retrieval_reference="synthetic:dataset-walkthrough",
        ts_received=acquired_at,
        ts_ingested=acquired_at,
    )
    store.publish(source, descriptor=origin, request=request)
    selections.append(
        DatasetInput(
            acquisition=AcquisitionManifest(run=run, completed=(origin,)),
            policy=replace(policy, max_gap=timedelta(minutes=60 - 30 * index)),
            eligibility=replace(
                eligibility,
                assertions=tuple(
                    row
                    for row in eligibility.assertions
                    if row.instrument_id == instrument.instrument_id
                ),
            ),
        )
    )
inputs = tuple(selections)

## 6. Publish, then reopen by identity

One `publish_dataset` call performs admission, assembly and atomic publication.
Its callbacks expose each quality report and publication progress. These sparse
archives remain **exploratory degraded**: known duplicates are retained as lineage,
while missing historical availability and coverage remain explicit limitations.

In [ ]:
admission, progress = {}, {}
snapshot_id = publish_dataset(
    inputs,
    store,
    destination,
    max_bytes=max_bytes,
    report=admission.__setitem__,
    progress=progress.__setitem__,
)
assert tuple(progress) == ("started", "complete")
assert all(manifest.identity == snapshot_id for manifest in progress.values())
assert [outcome.duplicate_count for outcome in admission.values()] == [1, 0]
NotebookDisplay.show_table(
    ("Input", "Source / stream", "Scope", "Invalid / gaps / duplicates"),
    [
        (
            selected.acquisition.run.instruments[0].symbol,
            (selected.acquisition.run.instruments[0].source, selected.acquisition.run.job.stream),
            outcome.scope,
            (outcome.invalid_count, outcome.gap_count, outcome.duplicate_count),
        )
        for selected, outcome in zip(inputs, admission.values(), strict=True)
    ],
)
print("Published snapshot:", snapshot_id)
print("Retained limitations:", "; ".join(next(iter(admission.values())).reasons))

In [ ]:
reopened = snapshots.read(snapshot_id, max_bytes=max_bytes)
assert len(reopened.partitions) == 2
assert tuple(part.policy for part in reopened.partitions) == tuple(item.policy for item in inputs)
assert tuple(part.eligibility for part in reopened.partitions) == tuple(
    item.eligibility for item in inputs
)
# Exact repeated selections are reused; the two instrument bindings remain separate.
assert publish_dataset((*inputs, inputs[0]), store, destination, max_bytes=max_bytes) == snapshot_id
precedence = tuple(
    (
        item.acquisition.run.instruments[0].source,
        item.acquisition.run.job.stream,
        item.acquisition.run.instruments[0].instrument_id,
    )
    for item in inputs
)
reader = AsOfSelector.read(
    destination, store, identity=snapshot_id, max_bytes=max_bytes, precedence=precedence
)
selected_events = reader.select(
    tuple(binding[2] for binding in precedence),
    start=start,
    end=archive_run.job.end,
    known_at=start + timedelta(seconds=1),
)
assert selected_events == ()
assert all(b"original trade availability unknown" in payload for payload in reader.history)
NotebookDisplay.show_table(
    ("Reopened input", "Canonical SHA-256", "Gap tolerance"),
    [
        (part.acquisition.run.instruments[0].symbol, part.sha256, part.policy.max_gap)
        for part in reopened.partitions
    ],
)
print("Known archive events selected:", len(selected_events))

The explicit snapshot ID reopens the raw bytes, canonical partitions and retained
evidence. An exact repeat yields the same identity. Both instrument partitions
survive even though their synthetic trade IDs match.

The cutoff query selects **zero** archive events because their original availability
is unknown. Download time does not establish historical knowledge, and an empty
selection does not prove a quiet or complete market. Full histories remain in
`reader.history`; those bytes include later information and are not a cutoff view.

### The same workflow from the command line

Serialize the same `DatasetInput` records and pass them to `alpha-dataset publish
--inputs`. This invocation runs outside the checkout and must return the same ID
as Python. For operating retained stores, see the [dataset API guide](../src/alpha_research/data/README.md).

In [ ]:
# standard library imports
from shutil import which

executable = which("alpha-dataset")
assert executable is not None, "Refresh the installed package first."
input_paths = [local / f"input-{index}.json" for index in range(len(inputs))]
for path, item in zip(input_paths, inputs, strict=True):
    path.write_bytes(codec.encode_json(item))
command = subprocess.run(
    [
        executable,
        "publish",
        "--store",
        str(raw_directory),
        "--snapshots",
        str(destination),
        "--max-bytes",
        str(max_bytes),
        "--inputs",
        *(str(path) for path in input_paths),
    ],
    cwd=local,
    capture_output=True,
    text=True,
    check=False,
    timeout=30,
)
assert command.returncode == 0, (command.stdout, command.stderr)
records = [json.loads(line) for line in command.stdout.splitlines()]
assert records[-1]["status"] == "complete" and records[-1]["snapshot_id"] == snapshot_id
NotebookDisplay.show_table(
    ("Stage", "Status", "Scope"),
    [(row["stage"], row["status"], row.get("scope", "")) for row in records],
)

### Exercise: what changes a dataset's identity?

Predict the effect of tightening only the first input's gap tolerance from one hour
to five minutes. Should its canonical partition change? Should the second partition
or the earlier snapshot change? Run the next cell to check your answer.

In [ ]:
changed_inputs = (
    replace(inputs[0], policy=replace(inputs[0].policy, max_gap=timedelta(minutes=5))),
    inputs[1],
)
changed_id = publish_dataset(changed_inputs, store, destination, max_bytes=max_bytes)
changed = snapshots.read(changed_id, max_bytes=max_bytes)
assert changed_id != snapshot_id
assert changed.partitions[0].sha256 != reopened.partitions[0].sha256
assert changed.partitions[1] == reopened.partitions[1]
assert snapshots.read(snapshot_id, max_bytes=max_bytes) == reopened
NotebookDisplay.show_table(
    ("Comparison", "Outcome"),
    [
        ("Dataset and first partition", "New identities"),
        ("Second partition", "Unchanged"),
        ("Earlier snapshot", "Still readable by its ID"),
    ],
)

The answer: policy is part of the evidence, so changing it changes identity; the
other input and earlier snapshot remain intact. Two representative failures below
show the boundary: causal use cannot accept unknown required coverage, and readers
require an immutable ID. Detailed corruption and interruption cases live in the
[dataset tests](../tests/unit/data/test_datasets.py),
[snapshot tests](../tests/unit/data/test_snapshots.py) and
[command tests](../tests/unit/pipelines/test_dataset_command.py).

In [ ]:
rejections = []
try:
    publish_dataset(
        (replace(inputs[0], policy=replace(policy, use="causal")),),
        store,
        destination,
        max_bytes=max_bytes,
    )
except ValueError as error:
    rejections.append(("Causal publication", str(error)))
else:
    raise AssertionError("Accepted unknown required coverage")
try:
    snapshots.read("latest", max_bytes=max_bytes)
except ValueError as error:
    rejections.append(("Mutable snapshot alias", str(error)))
else:
    raise AssertionError("Accepted mutable snapshot alias")
NotebookDisplay.show_table(("Refused request", "Reason"), rejections)

## 7. Select what was known at a cutoff

To see a nonempty cutoff selection, use a **synthetic captured trade** with an
original receipt clock, plus an explicitly evidenced correction. This fixture is
admitted in memory; a persisted captured-input adapter is not implemented here.

The correction is published at second 9 and available at second 10. A separate
synthetic membership correction becomes known at noon and withdraws the earlier
listing. Both histories are retained. This demonstrates the MR 09–17 semantics
without assigning invented receipt times to the archive example.

In [ ]:
# project library imports
from alpha_research.data.normalization import Occurrence
from alpha_research.data.revisions import Correction, TradeRevision
from alpha_research.pipelines.dataset import inspect_fixture

captured_catalog = codec.decode_json(
    (root / "configs/reference/binance-spot.json").read_bytes(), CatalogSnapshot
)
captured_run = Configuration.read(root / "configs/data-jobs/fixture-inspection.json").resolve(
    "btc-fixture", captured_catalog
)
original_line = (root / "tests/fixtures/m0/btcusdt.txt").read_bytes()
versions = []
for raw in (original_line, original_line.replace(b'"p":"67540.01000000"', b'"p":"67541.01000000"')):
    frame = local / "frame.txt"
    frame.write_bytes(raw)
    binding, event, digest = inspect_fixture(
        captured_run,
        frame,
        local_received=acquired_at,
        local_ingested=acquired_at,
        source_sequence=0,
    )
    origin = RawObject.from_file(
        frame,
        request=binding,
        retrieval_reference="synthetic:cutoff-walkthrough",
        ts_received=acquired_at,
        ts_ingested=acquired_at,
    )
    assert digest == origin.sha256
    versions.append(Occurrence(request=binding, origin=origin, position=0, raw=raw, event=event))
original_revision = TradeRevision(occurrences=(versions[0],))
correction = Correction(
    supersedes=original_revision.identity,
    replacement=TradeRevision(occurrences=(versions[1],)).identity,
    ts_published=start + timedelta(seconds=9),
    ts_available=start + timedelta(seconds=10),
    evidence_reference="synthetic:cutoff-walkthrough:correction",
)
captured_eligibility = replace(eligibility, catalog=captured_catalog)
report = policy.validate(
    binding,
    versions,
    eligibility=captured_eligibility,
    corrections=(correction,),
)
selector = AsOfSelector(
    (report.require_canonical(),),
    precedence=(
        (binding.instrument.source, binding.manifest.job.stream, binding.instrument.instrument_id),
    ),
)

In [ ]:
cutoffs = (
    start,
    start + timedelta(seconds=1),
    correction.ts_published,
    correction.ts_available,
    start + timedelta(hours=12),
)
cutoff_rows = []
for cutoff in cutoffs:
    visible = selector.select(
        (binding.instrument.instrument_id,),
        start=start,
        end=binding.end,
        known_at=cutoff,
    )
    cutoff_rows.append(
        (
            cutoff,
            [str(row.occurrences[0].event.price) for row in visible],
            [row.ts_knowledge for row in visible],
        )
    )
assert [row[1] for row in cutoff_rows] == [
    [],
    ["67540.01000000"],
    ["67540.01000000"],
    ["67541.01000000"],
    [],
]
assert selector.select(
    (binding.instrument.instrument_id,),
    start=start,
    end=binding.end,
    known_at=start + timedelta(seconds=1),
) == (original_revision,)
assert selector.history == (report.require_canonical(),)
NotebookDisplay.show_table(
    ("Known at", "Selected price", "Effective knowledge"),
    [
        (
            cutoff.isoformat(),
            ", ".join(prices) or "Unavailable",
            ", ".join(clock.isoformat() for clock in clocks) or "—",
        )
        for cutoff, prices, clocks in cutoff_rows
    ],
)
print("Retained quality scope:", report.scope)

The old price remains visible at second 9; the replacement enters at second 10.
At noon, the newly known membership correction removes the event from the selected
universe. Repeating the earlier query still returns the original revision. Validity
and knowledge are separate clocks; using the final revision everywhere would leak
future information into earlier decisions.

This is a bounded reference selector over individual trades. It establishes
software behavior, not empirical historical eligibility or complete source coverage.
More cases—equal-time ordering, binding-scoped disconnects and invalid queries—are
retained in the [as-of tests](../tests/unit/data/test_asof.py) and
[installed consumer](../tests/fixtures/mr17/asof_consumer.py). Historical NB 09–17 evidence
remains in the [MR records](../docs/roadmap/README.md).

For a new book sample use [capture](market-data-capture.ipynb); for unit, timing and
storage failure experiments use the [offline lab](foundation-contract-checks.ipynb).
Section 8 applies these contracts to MR 19 bounded queries. LOB adapters remain MR 25.

## 8. Read bounded query batches without losing controls

**Question:** can I request just prices while retaining the information needed to
interpret them? NB 19 reuses the published archive ID from section 6, then extends
the captured synthetic example with BTC and ETH disconnects. Only BTC is selected.

The reference reader enforces an aggregate canonical-byte budget and then batches
its in-memory selection. Batch size is an output bound, not a process-memory limit.
The installed acceptance [record](../docs/roadmap/mr-19-implementation.md) measures
actual file opens and Python allocations on a larger synthetic snapshot.


In [ ]:
# standard library imports
from threading import Event

# project library imports
from alpha_research.data.providers.tardis.decoder import TardisDecoder
from alpha_research.data.query import QueryMarketRow, QueryReader
from alpha_research.data.venues.binance.payloads import BinancePayloadDecoder

archive_query = QueryReader.read(
    destination,
    store,
    identity=snapshot_id,
    max_bytes=max_bytes,
    precedence=precedence,
)
archive_batches = list(
    archive_query.batches(
        tuple(item[2] for item in precedence),
        start=start,
        end=start + timedelta(hours=1),
        known_at=start + timedelta(hours=1),
        columns=("price",),
        batch_size=1,
    )
)
assert len(archive_batches) == 1 and archive_batches[0].rows == ()
assert len(archive_batches[0].evidence) == 2
print("Published archive query:", archive_batches[0].identity)
print(
    "Rows:",
    0,
    "; quality:",
    archive_batches[0].evidence[0].scope,
    "; original availability remains unknown",
)

eth_run = Configuration.read(root / "configs/data-jobs/fixture-inspection.json").resolve(
    "eth-fixture",
    captured_catalog,
)
eth_frame = root / "tests/fixtures/m0/ethusdt.txt"
eth_binding, eth_event, _ = inspect_fixture(
    eth_run,
    eth_frame,
    local_received=acquired_at,
    local_ingested=acquired_at,
    source_sequence=0,
)
eth_origin = RawObject.from_file(
    eth_frame,
    request=eth_binding,
    retrieval_reference="synthetic:mr19:eth",
    ts_received=acquired_at,
    ts_ingested=acquired_at,
)
eth_item = Occurrence(
    request=eth_binding,
    origin=eth_origin,
    position=0,
    raw=eth_frame.read_bytes(),
    event=eth_event,
)
query_payloads, query_precedence = [], []
for request, observations, corrections in (
    (binding, tuple(versions), (correction,)),
    (eth_binding, (eth_item,), ()),
):
    frame = local / "disconnect.txt"
    frame.write_bytes(b"\n")
    origin = RawObject.from_file(
        frame,
        request=request,
        retrieval_reference="synthetic:mr19:disconnect",
        ts_received=acquired_at,
        ts_ingested=acquired_at,
    )
    decoder = TardisDecoder(
        BinancePayloadDecoder(request.manifest.instruments), request.instrument.source
    )
    disconnect = decoder.decode_line(
        "",
        source_sequence=0,
        local_received=acquired_at,
        local_ingested=acquired_at,
        source_digest=origin.sha256,
        disconnect_capture_time=start + timedelta(seconds=2),
    )
    control = Occurrence(request=request, origin=origin, position=0, raw=b"\n", event=disconnect)
    quality = policy.validate(
        request,
        (*observations, control),
        eligibility=captured_eligibility,
        corrections=corrections,
    )
    query_payloads.append(quality.require_canonical())
    query_precedence.append(
        (request.instrument.source, request.manifest.job.stream, request.instrument.instrument_id)
    )
query = QueryReader(tuple(query_payloads), precedence=tuple(query_precedence), max_bytes=max_bytes)

In [ ]:
query_results = []
for seconds in (1, 9, 10, 43_200):
    batches = list(
        query.batches(
            (binding.instrument.instrument_id,),
            start=start,
            end=binding.end,
            known_at=start + timedelta(seconds=seconds),
            columns=("price",),
            batch_size=1,
        )
    )
    market = [row for batch in batches for row in batch.rows if isinstance(row, QueryMarketRow)]
    controls = [row for batch in batches for row in batch.rows if isinstance(row, Occurrence)]
    assert all(row.request == binding for row in controls)
    assert all(len(batch.rows) <= 1 and len(batch.evidence) == 1 for batch in batches)
    query_results.append((seconds, [str(row.values[0]) for row in market], len(controls)))
assert query_results == [
    (1, ["67540.01000000"], 0),
    (9, ["67540.01000000"], 1),
    (10, ["67541.01000000"], 1),
    (43_200, [], 1),
]
NotebookDisplay.show_table(
    ("Known at (seconds)", "Price projection", "BTC controls retained"), query_results
)
large_batches = list(
    query.batches(
        (binding.instrument.instrument_id,),
        start=start,
        end=binding.end,
        known_at=start + timedelta(hours=12),
        columns=("price",),
        batch_size=8,
    )
)
assert large_batches[0].identity == batches[0].identity
assert tuple(row for batch in large_batches for row in batch.rows) == tuple(
    row for batch in batches for row in batch.rows
)
print("Same query identity for batch sizes 1 and 8:", batches[0].identity)

The BTC disconnect survives price-only projection; the unrelated ETH disconnect
is absent. The correction appears only at second 10. At noon, membership removes
the market row while the applicable control and degraded evidence remain visible.

**Replay needs history:** the next query retains the original and corrected values,
with separate availability and membership evidence. These are replay candidates,
not selected economic inputs. Section 10 applies MR 20 chronology. Consumers still use the cutoff selector
at the step clock for membership; history is not permission to trade.


In [ ]:
history_batches = list(
    query.batches(
        (binding.instrument.instrument_id,),
        start=start,
        end=binding.end,
        known_at=start + timedelta(hours=12),
        columns=("revision",),
        batch_size=1,
        mode="history",
    )
)
projected = history_batches[0].rows[0]
assert isinstance(projected, QueryMarketRow)
head = projected.values[0]
assert isinstance(head, TradeRevision) and head.previous == original_revision
history_rows = []
while head is not None:
    history_rows.append((str(head.occurrences[0].event.price), head.ts_knowledge, head.identity))
    head = head.previous
NotebookDisplay.show_table(
    ("Retained price", "Effective availability", "Revision ID"), history_rows[::-1]
)
print("Membership assertions retained:", len(history_batches[0].evidence[0].eligibility.assertions))

stop = Event()
stream = query.batches(
    (binding.instrument.instrument_id,),
    start=start,
    end=binding.end,
    known_at=start + timedelta(seconds=10),
    columns=("price",),
    batch_size=1,
    cancelled=stop.is_set,
)
first_batch = next(stream)
stop.set()
try:
    next(stream)
except InterruptedError as error:
    print("After one bounded batch:", str(error))
else:
    raise AssertionError("Cancellation was ignored")
assert list(stream) == []

In [ ]:
failures = []
for label, options in (
    ("Zero batch size", {"batch_size": 0}),
    ("History without revision", {"mode": "history"}),
    ("Naive cutoff", {"known_at": datetime(2024, 6, 1)}),
):
    arguments = dict(
        start=start,
        end=binding.end,
        known_at=start + timedelta(seconds=10),
        columns=("price",),
        batch_size=1,
    )
    arguments.update(options)
    try:
        list(query.batches((binding.instrument.instrument_id,), **arguments))
    except (ValueError, TypeError) as error:
        failures.append((label, str(error)))
    else:
        raise AssertionError(f"Accepted {label}")
try:
    QueryReader(tuple(query_payloads), precedence=tuple(query_precedence), max_bytes=1)
except ValueError as error:
    failures.append(("Aggregate byte budget", str(error)))
else:
    raise AssertionError("Accepted excessive reference input")
try:
    QueryReader.read(
        destination, store, identity="latest", max_bytes=max_bytes, precedence=precedence
    )
except ValueError as error:
    failures.append(("Mutable snapshot alias", str(error)))
else:
    raise AssertionError("Accepted mutable alias")
NotebookDisplay.show_table(("Required refusal", "Reason"), failures)
assert len(failures) == 5

This checkpoint proves bounded output, explicit reference-input limits, cutoff
filtering before projection, retained correction chains, scoped controls and
cooperative cancellation. It does not prove complete market coverage or historical
eligibility. Canonical input, decoded selection and evidence remain in memory;
requests exceeding the explicit input budget fail instead of silently spilling.

Section 9 evaluates MR 18 using a separate retained permitted trade sample.
The synthetic failures above remain correctness evidence, not latency evidence.


## 9. Does this workload need another query backend?

**Question:** can one researcher inspect a minute and an hour of retained trades
within the agreed offline budgets? NB 18 uses **representative saved Binance trades**,
separate from the synthetic correctness examples in sections 5–8 and the real books
in sections 2–4. The source remains exploratory degraded: history has observations,
while the cutoff has none because historical availability and membership are unknown.

The [MR 18 record](../docs/roadmap/mr-18-implementation.md) fixes the workload before
measurement: 27,480 trades / 88,792,611 canonical bytes, one serial researcher,
256-row batches, three fresh processes and ten repetitions per query per process.
The accepted proposed budgets are 300 s verified open, 30 s query p95 and 1 GiB peak
process RSS. They do not qualify the roadmap's separate depth-query scale target.

The next cell recomputes the decision from the small **retained measurement record**;
it does not rerun the expensive benchmark inside this notebook. Check the timestamp
and hardware before applying these numbers elsewhere. The measured runtime is verified at its retained Git revision; each measured
file must still match the installed candidate. The new replay module is outside
this historical measurement, which makes no claim about replay performance. [Reproduction commands](notebook-operations.md#mr-18-query-workload)
run the real sample outside the checkout; the notebook's live synthetic queries,
exact correction outputs and required failure examples above still run afresh.


In [ ]:
# standard library imports
from math import ceil

measurement = json.loads((root / "docs/roadmap/mr-18-measurements.json").read_text())
measured_commit = measurement["runtime_commit"]
measured_paths = subprocess.check_output(
    ["git", "ls-tree", "-r", "--name-only", measured_commit, "src/alpha_research"],
    cwd=root,
    text=True,
).splitlines()
measured_digest = hashlib.sha256()
for name in measured_paths:
    relative = Path(name).relative_to("src/alpha_research")
    if relative.suffix not in {".py", ".json", ".typed"}:
        continue
    measured_bytes = subprocess.check_output(["git", "show", f"{measured_commit}:{name}"], cwd=root)
    measured_digest.update(str(relative).encode() + b"\0" + measured_bytes + b"\0")
    assert measured_bytes == (installed / relative).read_bytes(), f"Changed measured file: {name}"
assert measured_digest.hexdigest() == measurement["runtime_sha256"]
print("Historical query measurement verified at:", measured_commit)
print("Replay is additional unmeasured runtime; these are not replay performance results.")
assert (
    measurement["driver_sha256"]
    == hashlib.sha256((root / "examples/query_workload.py").read_bytes()).hexdigest()
)
assert measurement["lock_sha256"] == identity["uv_lock_sha256"]
assert len(measurement["runs"]) == 3
print(
    "Original reference failed; first load peak bytes:", measurement["baseline"]["peak_rss_bytes"]
)
print("Original minute query seconds:", measurement["baseline"]["minute_seconds"])
print("Original repetition plan stopped early; no baseline p95 is claimed.")
measured_rows = []
query_passes = []
for scenario in ("minute_history", "hour_history", "hour_cutoff"):
    samples = sorted(
        seconds for run in measurement["runs"] for seconds in run["queries"][scenario]["seconds"]
    )
    assert len(samples) == 30
    p50, p95, p99 = (samples[ceil(p * len(samples)) - 1] for p in (0.5, 0.95, 0.99))
    result = measurement["runs"][0]["queries"][scenario]
    query_passes.append(p95 <= 30)
    measured_rows.append((scenario, result["rows"], result["batches"], p50, p95, p99))
NotebookDisplay.show_table(
    ("Query", "Rows", "Batches", "p50 seconds", "p95 seconds", "p99 seconds"), measured_rows
)
loads = [run["load_seconds"] for run in measurement["runs"]]
peak_rss = max(run["peak_rss_bytes"] for run in measurement["runs"])
meets_budget = all(query_passes) and max(loads) <= 300 and peak_rss <= 1_073_741_824
print("Verified open seconds:", loads)
print("Peak process RSS bytes:", peak_rss)
print("Reference reader with admitted-partition reuse meets the workload:", meets_budget)
assert meets_budget, "The retained result does not justify backend deferral"
print("Measured at:", measurement["measured_at"], "; hardware:", measurement["hardware"])
print("Snapshot:", measurement["snapshot_id"])
NotebookDisplay.show_table(
    ("File role", "Opens", "Read calls", "Application bytes read"),
    [(role, *counts) for role, counts in measurement["load_reads"].items()],
)
print("Additional query file reads:", measurement["query_additional_reads"])
print("Parquet candidate bytes:", measurement["columnar"]["parquet_bytes"])
print("Parquet scan samples per query:", measurement["columnar"]["samples_per_query"])
print("Native scans exclude verified loading and full query semantics; no dependency admitted.")

In [ ]:
# third party library imports
from msgspec.structs import force_setattr

# Deliberately corrupt only a returned synthetic policy, bypassing its frozen API.
returned_policy = history_batches[0].evidence[0].policy
original_gap = returned_policy.max_gap
force_setattr(returned_policy, "max_gap", timedelta(0))
try:
    returned_policy.__post_init__()
except ValueError as error:
    print("Corrupted returned policy refuses validation:", str(error))
else:
    raise AssertionError("Invalid policy unexpectedly validated")
isolated_history = list(
    query.batches(
        (binding.instrument.instrument_id,),
        start=start,
        end=binding.end,
        known_at=start + timedelta(hours=12),
        columns=("revision",),
        batch_size=1,
        mode="history",
    )
)
assert isolated_history[0].evidence[0].policy.max_gap == original_gap
assert isolated_history[0].identity == history_batches[0].identity
assert isolated_history[0].rows == history_batches[0].rows
print("Cached admission survives returned-policy corruption; same query ID:")
print(isolated_history[0].identity)

The cutoff's empty result is an information limit, not evidence that no trades
occurred. History preserves all selected revisions and their unknown clocks. Verified opening still reads the whole snapshot and original raw input.
The optimized reference reuses admitted partitions; the minute query does not
repeat admission. Smaller result size still does not reduce verified-open I/O. Process RSS includes native allocations and
allocator retention; batch size is not an RSS ceiling.

Reader-cold measurements use fresh processes with an uncontrolled/warm OS file
cache. The first query of each kind and later calls are retained separately in the
record. Application reads include repeated verification and raw ZIP access; they
do not measure physical disk traffic. Thirty observations cannot qualify rare tails.

The optimized reference meets the agreed workload, so the columnar backend is
**deferred**. The record preserves the slower original reader and failed prototype
attempts. Reusing admitted partitions and copying outputs avoids a new runtime
dependency while retaining the query and integrity contracts.

Use the explicit decision and limitations in the [MR 18 record](../docs/roadmap/mr-18-implementation.md).
A stricter latency/memory budget, larger retained interval, known-availability data,
depth adapter or concurrent consumer requires a new measured decision. The next
roadmap MR is **20 — Replay with shared event stepping**; this checkpoint does not
implement it or change the frozen P1 study.


## 10. What becomes known at each replay step?

**Question:** does a later-known correction or gap change the earlier trace?
NB 20 uses the same synthetic captured BTC trade, scoped disconnect and late
correction as section 8. A configured wakeup at second 0 and an additional
fill-eligibility/expiry wakeup at second 1.5 are illustrative scheduled times;
this checkpoint computes no orders, fills or P&L.

`Replay` reads bounded retained history, orders it by effective knowledge and
logical source position, then advances to the next source, requested wakeup or
terminal time. The callback admits every equal-time source before expiry, fill,
decision and mark. A callback performs only operations due at that time.


In [ ]:
# project library imports
from alpha_research.data.asof import PointEvent
from alpha_research.data.events import Disconnect
from alpha_research.data.replay import Replay, ReplayCursor, ReplayPhase


class ReplayTrace:
    """Observe source availability and phase order without implementing economics."""

    def __init__(self) -> None:
        """Start an independent trace for this consumer."""
        self.rows: list[tuple[float, ReplayPhase, str]] = []

    def consume(self, phase: ReplayPhase, instant: datetime, item: PointEvent | None) -> None:
        """Record the then-available payload and current phase.

        Args:
            phase: Source admission or the scheduled operation phase.
            instant: Current UTC clock.
            item: Available source event, or None for later phases.
        """
        value = ""
        if isinstance(item, TradeRevision):
            event = item.occurrences[0].event
            assert not isinstance(event, Disconnect)
            value = str(event.price)
        elif isinstance(item, Occurrence):
            value = "scoped disconnect"
        self.rows.append(((instant - start).total_seconds(), phase, value))


replay_options = dict(
    start=start,
    end=binding.end,
    terminal=start + timedelta(seconds=12),
    config_identity=captured_run.identity,
)
replay_instruments = (binding.instrument.instrument_id,)
traces, completed_cursors = [], []
for size in (1, 8):
    engine = Replay(query, replay_instruments, batch_size=size, **replay_options)
    assert engine.cursor is None, "Reading ahead advanced the replay clock"
    trace = ReplayTrace()
    engine.step(start, trace.consume)
    engine.step(start + timedelta(seconds=1.5), trace.consume)
    engine.step(start + timedelta(seconds=1.5), trace.consume)
    while engine.step(replay_options["terminal"], trace.consume) is not None:
        pass
    traces.append(trace.rows)
    completed_cursors.append(engine.cursor)
assert traces[0] == traces[1] and completed_cursors[0] == completed_cursors[1]
assert [row for row in traces[0] if row[1] == "source"] == [
    (1.0, "source", "67540.01000000"),
    (2.0, "source", "scoped disconnect"),
    (10.0, "source", "67541.01000000"),
]
NotebookDisplay.show_table(("Seconds", "Phase", "Available value"), traces[0])
print("Batch sizes 1/8: identical ordered trace and completed cursor")
print("Quality:", engine.evidence[0].scope, "; membership remains cutoff-dependent")

The old price appears at second 1, the gap at second 2 and the correction at
second 10. Read-ahead did not advance any of those clocks. The extra 1.5-second
step shows how a consumer can request a time between source events.

**Resume question:** can the trace continue from second 1 using a different batch
size? Save the completed cursor with the consumer's trace/state and outstanding
schedule. The cursor binds selected inputs, timing/configuration identity and
ordering-policy version; it does not contain an economic ledger or pending intent.


In [ ]:
engine = Replay(query, replay_instruments, batch_size=1, **replay_options)
prefix = ReplayTrace()
engine.step(start, prefix.consume)
checkpoint = engine.step(start + timedelta(seconds=1.5), prefix.consume)
assert checkpoint is not None and checkpoint.clock == start + timedelta(seconds=1)
checkpoint_bytes = codec.encode_json(checkpoint)
restored = codec.decode_json(checkpoint_bytes, ReplayCursor)
resumed = Replay(query, replay_instruments, batch_size=8, cursor=restored, **replay_options)
resumed.step(start + timedelta(seconds=1.5), prefix.consume)
while resumed.step(replay_options["terminal"], prefix.consume) is not None:
    pass
assert prefix.rows == traces[0] and resumed.cursor == completed_cursors[0]
print("Saved completed-step cursor:", checkpoint_bytes.decode())
print("Restored consumer prefix + scheduled wakeup + replay suffix matches uninterrupted trace")

failures = []
for label, options in (
    ("Changed consumer configuration", {"config_identity": "b" * 64}),
    ("Changed terminal/input query", {"terminal": start + timedelta(seconds=11)}),
):
    try:
        Replay(
            query, replay_instruments, batch_size=1, cursor=restored, **(replay_options | options)
        )
    except ValueError as error:
        failures.append((label, str(error)))
    else:
        raise AssertionError(f"Accepted {label}")
engine = Replay(query, replay_instruments, batch_size=1, cursor=restored, **replay_options)
for label, instant in (
    ("Repeated completed time", restored.clock),
    ("Naive wakeup", datetime(2024, 6, 1)),
):
    try:
        engine.step(instant, ReplayTrace().consume)
    except (ValueError, TypeError) as error:
        failures.append((label, str(error)))
    else:
        raise AssertionError(f"Accepted {label}")
NotebookDisplay.show_table(("Required refusal", "Reason"), failures)
assert len(failures) == 4

This proves source/phase and cursor parity for synthetic trade/control history.
Regression tests additionally split an equal-time trade/control group across
one-row batches, perturb only future rows/hashes, reject incomplete positions,
and refuse continuation/checkpointing after callback failure. Unknown archive
availability remains unschedulable; membership/quality must still be evaluated
at the consumer's current cutoff.

The reference retains all byte-capped inputs and an ordered index in memory;
batch size is a delivery bound, not a process-memory limit. No L2/L3 adapter,
accepted-book checkpoint, strategy economics or durable account recovery is
introduced. E1/MR 44 will prove earlier fills survive later-known gaps, exact
expiry rejects fills, and resumed economics agree. **Next: MR 21**, the EDA and
extension workflow. This notebook does not start it.


In [ ]:
# All synthetic files belong to this run; the saved book inputs were only read.
workspace.cleanup()